In [ ]:
pip install langchain-docling

In [ ]:
import ollama
from langchain_docling.loader import DoclingLoader
model = "llama3:8b"

FILE_PATH = r"C:\Users\ADMIN\Downloads\FDS LAB MANUAL (1).docx"
loader = DoclingLoader(file_path=FILE_PATH)
docs = loader.load()
print(docs)

In [ ]:
pip install langchain-text-splitters

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=60)
text = text_splitter.split_documents(docs)

In [ ]:
print(text[0])
print(text[1])

model = "BAAI/bge-large-en-v1.5"

In [ ]:
pip install sentence-transformers

In [ ]:
pip install langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name = "BAAI/bge-large-en-v1.5")



In [ ]:
pip install langchain-chroma

In [10]:
from langchain_chroma import Chroma
vector_store=Chroma(
    collection_name="sample",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

In [ ]:
print(vector_store._collection.count())

In [ ]:
pip install langchain-community

In [22]:
from langchain_community.vectorstores.utils import filter_complex_metadata
text = filter_complex_metadata(text)

In [ ]:
text[0]

In [ ]:
vector_store.add_documents(text)

In [ ]:
vector_store._collection.count()

In [ ]:
vector_store._collection.get(ids=['e8f36f18-63af-4a85-bd9b-7ffd8144b2ee'],include=["embeddings","documents"])

In [ ]:
def ask_llm(prompt):
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        stream=True
    )

    for res in response:
        print(res["message"]["content"], end="", flush=True)



In [ ]:
while True:
    retriver = vector_store.as_retriever()
    query = input("Enter the query:")
    retrieved_docs= retriver.invoke(query)
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )
    print(context)
    prompt = f"""yo u are a rag assistant and you need to answer questions on provided contexts only.if answer is not available in provided context then gently respond " i don't know".

        The input is {query}
        The retrived context are {context}
    """
    ask_llm(finalprompt)